In [1]:
from __future__ import annotations

import json
import textwrap
import time
import zipfile
from datetime import datetime, timezone
from io import BytesIO
from uuid import uuid4

import pandas as pd
import vertexai
from google import genai
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from vertexai.language_models import TextEmbeddingModel

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "weaviate_mentor_rag_backend"

DOCUMENT_TABLE_ID = "weaviate_mentor_documents"
CHUNK_TABLE_ID = "weaviate_mentor_chunks"
TEST_ANSWER_TABLE_ID = "weaviate_mentor_test_answers"
BACKEND_ASSET_TABLE_ID = "weaviate_mentor_backend_assets"

TEXT_MODEL = "gemini-2.5-flash"
PLANNING_MODEL = "gemini-2.5-pro"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

CHUNK_SIZE_CHARS = 1000
CHUNK_OVERLAP_CHARS = 180
MAX_EMBEDDING_TEXT_CHARS = 3000
EMBEDDING_BATCH_SIZE = 16

TOP_K_DEFAULT = 8

ALLOWED_ORIGIN = "https://weaviate-voice-mentor.lovable.app"

GCS_SOURCE_PREFIX = "weaviate-mentor-rag/source"
GCS_BACKEND_PREFIX = "weaviate-mentor-rag/backend"
GCS_WIDGET_PREFIX = "weaviate-mentor-rag/widget"
GCS_SUMMARY_PREFIX = "weaviate-mentor-rag/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Dataset:", DATASET_ID)
print("Allowed origin:", ALLOWED_ORIGIN)
print("Text model:", TEXT_MODEL)
print("Embedding model:", TEXT_EMBEDDING_MODEL)

Configuration loaded.
Project: leafy-guide-497515-m4
Location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Dataset: weaviate_mentor_rag_backend
Allowed origin: https://weaviate-voice-mentor.lovable.app
Text model: gemini-2.5-flash
Embedding model: text-embedding-005


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

google_vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
bucket.reload()

bigquery_client = bigquery.Client(project=PROJECT_ID)

text_embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL
)

BUCKET_LOCATION = bucket.location

if BUCKET_LOCATION in {"US", "EU"}:
    BIGQUERY_LOCATION = BUCKET_LOCATION
else:
    BIGQUERY_LOCATION = BUCKET_LOCATION.lower()

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("Bucket location:", BUCKET_LOCATION)
print("BigQuery dataset location:", BIGQUERY_LOCATION)
print("BigQuery project:", bigquery_client.project)

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Clients created.
Bucket exists: True
Bucket location: EU
BigQuery dataset location: EU
BigQuery project: leafy-guide-497515-m4


In [4]:
dataset_ref = bigquery.Dataset(
    f"{PROJECT_ID}.{DATASET_ID}"
)

dataset_ref.location = BIGQUERY_LOCATION

try:
    dataset = bigquery_client.create_dataset(dataset_ref)

    print("Created dataset:", dataset.full_dataset_id)

except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)

    print("Dataset already exists:", dataset.full_dataset_id)

document_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{DOCUMENT_TABLE_ID}"
chunk_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
test_answer_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TEST_ANSWER_TABLE_ID}"
backend_asset_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{BACKEND_ASSET_TABLE_ID}"

print("Document table:", document_table_ref)
print("Chunk table:", chunk_table_ref)
print("Test answer table:", test_answer_table_ref)
print("Backend asset table:", backend_asset_table_ref)

Created dataset: leafy-guide-497515-m4:weaviate_mentor_rag_backend
Document table: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_documents
Chunk table: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_chunks
Test answer table: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_test_answers
Backend asset table: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_backend_assets


In [5]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)

        print("Table already exists:", table.full_table_id)

        return table

    except NotFound:
        print("Table does not exist. Creating:", table_ref)

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(table)

        print("Created table:", created_table.full_table_id)

        return created_table

In [6]:
document_schema = [
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("metadata_json", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

chunk_schema = [
    bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("global_chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("section_title", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("chunk_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_char_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("embedding_model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

test_answer_schema = [
    bigquery.SchemaField("answer_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("question", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("answer", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("used_chunk_ids", "STRING", mode="REPEATED"),
    bigquery.SchemaField("used_source_titles", "STRING", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

backend_asset_schema = [
    bigquery.SchemaField("asset_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("asset_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("filename", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("gcs_uri", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

In [7]:
document_table = ensure_bigquery_table(
    document_table_ref,
    document_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

test_answer_table = ensure_bigquery_table(
    test_answer_table_ref,
    test_answer_schema,
)

backend_asset_table = ensure_bigquery_table(
    backend_asset_table_ref,
    backend_asset_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_documents
Created table: leafy-guide-497515-m4:weaviate_mentor_rag_backend.weaviate_mentor_documents
Table does not exist. Creating: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_chunks
Created table: leafy-guide-497515-m4:weaviate_mentor_rag_backend.weaviate_mentor_chunks
Table does not exist. Creating: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_test_answers
Created table: leafy-guide-497515-m4:weaviate_mentor_rag_backend.weaviate_mentor_test_answers
Table does not exist. Creating: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_backend_assets
Created table: leafy-guide-497515-m4:weaviate_mentor_rag_backend.weaviate_mentor_backend_assets


In [8]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )


def upload_bytes_to_gcs(
    data: bytes,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        data,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )


def rows_to_ndjson(
    rows: list[dict],
) -> str:
    return "\n".join(
        json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        )
        for row in rows
    )

In [9]:
def batch_load_rows_to_bigquery(
    *,
    rows: list[dict],
    table_ref: str,
    schema: list[bigquery.SchemaField],
    table_name: str,
    run_id: str,
) -> dict:
    ndjson_text = rows_to_ndjson(rows)

    blob_name = (
        f"{GCS_SOURCE_PREFIX}/"
        f"{run_id}/"
        f"{table_name}.ndjson"
    )

    gcs_uri = upload_text_to_gcs(
        ndjson_text,
        blob_name=blob_name,
        content_type="application/x-ndjson",
    )

    source_bytes = len(ndjson_text.encode("utf-8"))

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )

    started_at = time.perf_counter()

    load_job = bigquery_client.load_table_from_uri(
        gcs_uri,
        table_ref,
        location=dataset.location,
        job_config=job_config,
    )

    load_job.result()

    load_seconds = time.perf_counter() - started_at

    destination_table = bigquery_client.get_table(table_ref)

    print("=" * 100)
    print("Loaded table:", table_name)
    print("Rows:", destination_table.num_rows)
    print("GCS URI:", gcs_uri)
    print("Bytes:", source_bytes)
    print("Seconds:", round(load_seconds, 4))

    return {
        "table_name": table_name,
        "gcs_uri": gcs_uri,
        "row_count": destination_table.num_rows,
        "source_bytes": source_bytes,
        "load_seconds": round(load_seconds, 4),
        "job_id": load_job.job_id,
    }

In [10]:
WEAVIATE_KNOWLEDGE_MARKDOWN = """
# Weaviate Voice Mentor Knowledge Base

## 1. What Weaviate is

Weaviate is an AI-native vector database. It stores objects, their properties, and vector embeddings. A useful mental model is that a Weaviate collection is similar to a database table, and an object is similar to a row, but every object can also have a vector representation for semantic search.

Weaviate is commonly used for:
- semantic search
- hybrid search
- RAG applications
- recommendation systems
- multimodal search
- document search
- knowledge assistants
- AI agents

The key idea is that text, images, or other data can be represented as vectors. Similar meanings are close together in vector space.

## 2. Collections

A collection is the main container for objects in Weaviate. Older materials may call this a class. A collection defines:
- the collection name
- properties
- data types
- vectorizer configuration
- generative module configuration
- reference properties
- optional indexing behavior

Example collections:
- Article
- Product
- HardwareReview
- Question
- ProjectDocument

A collection can be created with properties such as title, content, category, price, rating, or created_at.

## 3. Objects and properties

An object is one record inside a collection. It can contain scalar properties and optionally references to other objects.

Example Product object:
- sku
- name
- description
- category
- price

Example HardwareReview object:
- review_text
- rating
- reviewer
- reference to Product

When an object is inserted, Weaviate can automatically vectorize selected textual properties if a vectorizer module is configured. Alternatively, the application can provide vectors manually.

## 4. Vector embeddings

An embedding is a numerical vector that represents semantic meaning. For example, with OpenAI text-embedding-3-small the vector dimension is commonly 1536.

Vector embeddings let Weaviate answer questions like:
- which products are semantically similar to this query?
- which document chunk is closest to this user question?
- which review describes the same problem?
- which image looks similar to another image?

Vector length is not arbitrary. It depends on the embedding model.

## 5. BM25 keyword search

BM25 is keyword-based search. It is useful when the exact words matter.

BM25 is good for:
- exact product codes
- IDs
- names
- error codes
- rare terms
- explicit keywords

Example:
A query for `ORD-1234` or `PCIe reset` may work very well with BM25 because the exact token matters.

## 6. Vector search

Vector search is semantic search. It is useful when the query and the document use different words but mean similar things.

Example:
Query:
`laptop is slow when many apps are open`

Could match:
`The computer has insufficient RAM for multitasking workloads.`

Vector search understands meaning better than pure keyword search, but it may miss exact rare terms.

## 7. Hybrid search

Hybrid search combines BM25 keyword search and vector search.

This is often better than using only one of them because:
- BM25 catches exact terms
- vector search catches semantic similarity
- hybrid search balances both signals

In Weaviate, hybrid search is often useful for RAG, product search, documentation search, and support assistants.

The alpha parameter controls the balance:
- alpha close to 0 means more keyword/BM25 influence
- alpha close to 1 means more vector influence
- alpha around 0.5 balances both

## 8. RAG with Weaviate

RAG means Retrieval-Augmented Generation.

Typical RAG flow:
1. User asks a question.
2. Application searches Weaviate for relevant chunks.
3. Retrieved chunks are passed to an LLM.
4. LLM answers using only the retrieved context.
5. Sources are shown to the user.

Weaviate can be used as the retrieval layer in this architecture.

Important RAG rules:
- split documents into chunks
- store source metadata
- retrieve the most relevant chunks
- keep the answer grounded in sources
- say when the context is insufficient
- avoid inventing facts outside the retrieved context

## 9. Chunking

Chunking means splitting large documents into smaller pieces.

Good chunks should:
- be small enough for retrieval
- preserve meaning
- include metadata
- not cut important context too aggressively

Common chunk settings:
- 500 to 1500 characters
- 100 to 300 characters overlap
- split on paragraphs or sentence boundaries when possible

Chunk metadata can include:
- source file
- notebook name
- section title
- created_at
- document type
- repository path

## 10. Metadata filters

Metadata filters restrict search results.

Examples:
- search only in notebooks
- search only in project docs
- search only in a specific collection
- search only documents created after a date
- search only reviews with rating lower than 3

Filters are useful when the vector search finds semantically similar content from the wrong area.

## 11. References

References connect objects across collections.

Example:
A HardwareReview can reference a Product.

This allows queries like:
- find reviews for this product
- find products with negative reviews
- retrieve product data and review data together

References are useful when the domain is relational.

## 12. Weaviate Cloud vs local Docker

Local Docker is useful for experiments and learning. Weaviate Cloud is better for hosted projects and portfolio demos.

Local Docker:
- fast for learning
- easy to reset
- good for local experiments

Weaviate Cloud:
- hosted endpoint
- API key authentication
- easier public demo integration
- less local infrastructure maintenance

## 13. Python client pattern

Typical Python client flow:
1. import weaviate
2. connect to Weaviate Cloud or local instance
3. check client.is_ready()
4. create or access collection
5. insert objects
6. run BM25, vector, or hybrid queries
7. close the client

Closing the client does not delete data. It only closes the connection.

## 14. Common beginner mistakes

Common mistakes:
- wrong import path after client version changes
- forgetting to close the client
- confusing JSON string with Python dict
- confusing collection with object
- expecting vector search to work without vectors
- forgetting API keys
- using too large chunks
- mixing old v3 client syntax with v4 client syntax
- expecting Swagger CRUD JSON to be the same as vector database schema design

## 15. JSON, dict, and serialization

A Python dict is an in-memory Python object.

JSON is a text format.

Serialization means converting a Python object into a transportable or storable format such as a JSON string.

In APIs, JSON is commonly used because it is language-independent.

## 16. Weaviate and SQL mental model

SQL database:
- table
- row
- column
- foreign key
- SQL query

Weaviate:
- collection
- object
- property
- reference
- vector, BM25, hybrid query

The analogy is useful but not perfect. Weaviate is optimized for search and AI retrieval, not traditional transactional SQL workloads.

## 17. When to use Weaviate

Use Weaviate when the main problem involves:
- semantic search
- RAG
- recommendations
- unstructured documents
- multimodal search
- hybrid keyword plus vector retrieval

Do not use Weaviate as a replacement for every relational database. Many applications use both SQL and Weaviate together.

## 18. Voice mentor behavior

The Weaviate Voice Mentor should:
- answer in Polish by default
- use simple explanations
- use analogies to SQL tables and rows when helpful
- explain code step by step
- say when context is missing
- avoid pretending to know things not in the knowledge base
- focus on Weaviate, vector databases, embeddings, hybrid search, RAG, collections, objects, references, and Python usage
"""

EXTRA_WEAVIATE_MARKDOWN = """
"""

full_weaviate_markdown = "\n\n".join(
    [
        WEAVIATE_KNOWLEDGE_MARKDOWN.strip(),
        EXTRA_WEAVIATE_MARKDOWN.strip(),
    ]
).strip()

print("Knowledge base characters:", len(full_weaviate_markdown))
print(full_weaviate_markdown[:1200])

Knowledge base characters: 7681
# Weaviate Voice Mentor Knowledge Base

## 1. What Weaviate is

Weaviate is an AI-native vector database. It stores objects, their properties, and vector embeddings. A useful mental model is that a Weaviate collection is similar to a database table, and an object is similar to a row, but every object can also have a vector representation for semantic search.

Weaviate is commonly used for:
- semantic search
- hybrid search
- RAG applications
- recommendation systems
- multimodal search
- document search
- knowledge assistants
- AI agents

The key idea is that text, images, or other data can be represented as vectors. Similar meanings are close together in vector space.

## 2. Collections

A collection is the main container for objects in Weaviate. Older materials may call this a class. A collection defines:
- the collection name
- properties
- data types
- vectorizer configuration
- generative module configuration
- reference properties
- optional indexi

In [11]:
run_id = str(uuid4())
run_timestamp = datetime.now(timezone.utc)

knowledge_blob_name = (
    f"{GCS_SOURCE_PREFIX}/"
    f"{run_id}/"
    "weaviate_knowledge_base.md"
)

knowledge_gcs_uri = upload_text_to_gcs(
    full_weaviate_markdown,
    blob_name=knowledge_blob_name,
    content_type="text/markdown",
)

print("Run ID:", run_id)
print("Knowledge base saved to:")
print(knowledge_gcs_uri)

Run ID: b3c00ed4-5393-4261-bd1f-c1df30fe8cf7
Knowledge base saved to:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_knowledge_base.md


In [12]:
def split_markdown_into_sections(
    markdown_text: str,
) -> list[dict]:
    sections = []

    current_title = "Introduction"
    current_lines = []

    for line in markdown_text.splitlines():
        if line.startswith("## "):
            if current_lines:
                sections.append(
                    {
                        "section_title": current_title,
                        "content": "\n".join(current_lines).strip(),
                    }
                )

            current_title = line.replace("## ", "").strip()
            current_lines = [line]

        else:
            current_lines.append(line)

    if current_lines:
        sections.append(
            {
                "section_title": current_title,
                "content": "\n".join(current_lines).strip(),
            }
        )

    return [
        section
        for section in sections
        if section["content"]
    ]


weaviate_sections = split_markdown_into_sections(
    full_weaviate_markdown
)

print("Sections:", len(weaviate_sections))

pd.DataFrame(
    [
        {
            "section_title": section["section_title"],
            "content_length": len(section["content"]),
            "preview": section["content"][:160],
        }
        for section in weaviate_sections
    ]
).head(20)

Sections: 19


,section_title,content_length,preview
0,Introduction,38,# Weaviate Voice Mentor Knowledge Base
1,1. What Weaviate is,637,## 1. What Weaviate is\n\nWeaviate is an AI-na...
2,2. Collections,497,## 2. Collections\n\nA collection is the main ...
3,3. Objects and properties,504,## 3. Objects and properties\n\nAn object is o...
4,4. Vector embeddings,506,## 4. Vector embeddings\n\nAn embedding is a n...
5,5. BM25 keyword search,311,## 5. BM25 keyword search\n\nBM25 is keyword-b...
6,6. Vector search,382,## 6. Vector search\n\nVector search is semant...
7,7. Hybrid search,541,## 7. Hybrid search\n\nHybrid search combines ...
8,8. RAG with Weaviate,599,## 8. RAG with Weaviate\n\nRAG means Retrieval...
9,9. Chunking,475,## 9. Chunking\n\nChunking means splitting lar...


In [13]:
document_rows = []

for document_number, section in enumerate(
    weaviate_sections,
    start=1,
):
    document_rows.append(
        {
            "document_id": str(uuid4()),
            "document_number": document_number,
            "source_type": "weaviate_knowledge_markdown",
            "source_title": section["section_title"],
            "content": section["content"],
            "metadata_json": json.dumps(
                {
                    "knowledge_gcs_uri": knowledge_gcs_uri,
                    "run_id": run_id,
                    "section_title": section["section_title"],
                },
                ensure_ascii=False,
                default=str,
            ),
            "created_at": run_timestamp.isoformat(),
        }
    )

print("Document rows:", len(document_rows))

pd.DataFrame(
    [
        {
            "document_number": row["document_number"],
            "source_title": row["source_title"],
            "content_length": len(row["content"]),
        }
        for row in document_rows
    ]
).head(20)

Document rows: 19


,document_number,source_title,content_length
0,1,Introduction,38
1,2,1. What Weaviate is,637
2,3,2. Collections,497
3,4,3. Objects and properties,504
4,5,4. Vector embeddings,506
5,6,5. BM25 keyword search,311
6,7,6. Vector search,382
7,8,7. Hybrid search,541
8,9,8. RAG with Weaviate,599
9,10,9. Chunking,475


In [14]:
def normalize_text(
    text: str,
) -> str:
    return " ".join(text.split())


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(
            start + chunk_size_chars,
            len(clean_text),
        )

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(
                ". ",
                start,
                end,
            )

            if sentence_boundary > start + int(chunk_size_chars * 0.6):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(
            0,
            end - chunk_overlap_chars,
        )

    return chunks

In [15]:
chunk_rows_without_embeddings = []
global_chunk_number = 1

for document in document_rows:
    chunks = split_text_into_chunks(
        document["content"]
    )

    print("=" * 100)
    print("Document:", document["source_title"])
    print("Chunks:", len(chunks))

    for chunk_number, chunk_text in enumerate(
        chunks,
        start=1,
    ):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "document_id": document["document_id"],
                "document_number": document["document_number"],
                "chunk_number": chunk_number,
                "global_chunk_number": global_chunk_number,
                "source_type": document["source_type"],
                "source_title": document["source_title"],
                "section_title": document["source_title"],
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": TEXT_EMBEDDING_MODEL,
                "created_at": run_timestamp.isoformat(),
            }
        )

        global_chunk_number += 1

print("Chunks without embeddings:", len(chunk_rows_without_embeddings))

chunks_preview_df = pd.DataFrame(
    [
        {
            "global_chunk_number": row["global_chunk_number"],
            "source_title": row["source_title"],
            "chunk_number": row["chunk_number"],
            "chunk_char_count": row["chunk_char_count"],
            "preview": row["chunk_text"][:180],
        }
        for row in chunk_rows_without_embeddings
    ]
)

chunks_preview_df.head(20)

Document: Introduction
Chunks: 1
Document: 1. What Weaviate is
Chunks: 1
Document: 2. Collections
Chunks: 1
Document: 3. Objects and properties
Chunks: 1
Document: 4. Vector embeddings
Chunks: 1
Document: 5. BM25 keyword search
Chunks: 1
Document: 6. Vector search
Chunks: 1
Document: 7. Hybrid search
Chunks: 1
Document: 8. RAG with Weaviate
Chunks: 1
Document: 9. Chunking
Chunks: 1
Document: 10. Metadata filters
Chunks: 1
Document: 11. References
Chunks: 1
Document: 12. Weaviate Cloud vs local Docker
Chunks: 1
Document: 13. Python client pattern
Chunks: 1
Document: 14. Common beginner mistakes
Chunks: 1
Document: 15. JSON, dict, and serialization
Chunks: 1
Document: 16. Weaviate and SQL mental model
Chunks: 1
Document: 17. When to use Weaviate
Chunks: 1
Document: 18. Voice mentor behavior
Chunks: 1
Chunks without embeddings: 19


,global_chunk_number,source_title,chunk_number,chunk_char_count,preview
0,1,Introduction,1,38,# Weaviate Voice Mentor Knowledge Base
1,2,1. What Weaviate is,1,634,## 1. What Weaviate is Weaviate is an AI-nativ...
2,3,2. Collections,1,494,## 2. Collections A collection is the main con...
3,4,3. Objects and properties,1,500,## 3. Objects and properties An object is one ...
4,5,4. Vector embeddings,1,503,## 4. Vector embeddings An embedding is a nume...
5,6,5. BM25 keyword search,1,308,## 5. BM25 keyword search BM25 is keyword-base...
6,7,6. Vector search,1,378,## 6. Vector search Vector search is semantic ...
7,8,7. Hybrid search,1,537,## 7. Hybrid search Hybrid search combines BM2...
8,9,8. RAG with Weaviate,1,595,## 8. RAG with Weaviate RAG means Retrieval-Au...
9,10,9. Chunking,1,471,## 9. Chunking Chunking means splitting large ...


In [16]:
def shorten_text_for_embedding(
    text: str,
    *,
    max_chars: int = MAX_EMBEDDING_TEXT_CHARS,
) -> str:
    clean_text = normalize_text(text)

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_text_embeddings_in_batches(
    texts: list[str],
    *,
    batch_size: int = EMBEDDING_BATCH_SIZE,
) -> list[list[float]]:
    all_embeddings = []

    for start_index in range(
        0,
        len(texts),
        batch_size,
    ):
        end_index = min(
            start_index + batch_size,
            len(texts),
        )

        batch = [
            shorten_text_for_embedding(text)
            for text in texts[start_index:end_index]
        ]

        print(
            "Embedding batch:",
            start_index,
            "->",
            end_index - 1,
            "| size:",
            len(batch),
        )

        batch_embeddings = text_embedding_model.get_embeddings(
            batch
        )

        all_embeddings.extend(
            [
                list(embedding.values)
                for embedding in batch_embeddings
            ]
        )

    return all_embeddings

In [17]:
chunk_texts = [
    row["chunk_text"]
    for row in chunk_rows_without_embeddings
]

embedding_started_at = time.perf_counter()

chunk_embeddings = get_text_embeddings_in_batches(
    chunk_texts
)

embedding_seconds = time.perf_counter() - embedding_started_at

assert len(chunk_embeddings) == len(chunk_rows_without_embeddings)

chunk_rows = []

for row, embedding in zip(
    chunk_rows_without_embeddings,
    chunk_embeddings,
    strict=True,
):
    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

print("Chunks with embeddings:", len(chunk_rows))
print("Embedding seconds:", round(embedding_seconds, 2))
print("Embedding dimension:", len(chunk_rows[0]["embedding"]))

Embedding batch: 0 -> 15 | size: 16
Embedding batch: 16 -> 18 | size: 3
Chunks with embeddings: 19
Embedding seconds: 1.85
Embedding dimension: 768


In [18]:
document_load_result = batch_load_rows_to_bigquery(
    rows=document_rows,
    table_ref=document_table_ref,
    schema=document_schema,
    table_name=DOCUMENT_TABLE_ID,
    run_id=run_id,
)

chunk_load_result = batch_load_rows_to_bigquery(
    rows=chunk_rows,
    table_ref=chunk_table_ref,
    schema=chunk_schema,
    table_name=CHUNK_TABLE_ID,
    run_id=run_id,
)

print(document_load_result)
print(chunk_load_result)

Loaded table: weaviate_mentor_documents
Rows: 19
GCS URI: gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_documents.ndjson
Bytes: 17625
Seconds: 3.2715
Loaded table: weaviate_mentor_chunks
Rows: 19
GCS URI: gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_chunks.ndjson
Bytes: 339175
Seconds: 1.8596
{'table_name': 'weaviate_mentor_documents', 'gcs_uri': 'gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_documents.ndjson', 'row_count': 19, 'source_bytes': 17625, 'load_seconds': 3.2715, 'job_id': 'b977610e-581f-4211-8620-7c1372510d72'}
{'table_name': 'weaviate_mentor_chunks', 'gcs_uri': 'gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_chunks.ndjson', 'row_count': 19, 'source_bytes': 339175, 'load_seconds'

In [19]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{document_table_ref}`) AS document_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

verification_result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print("Document count:", verification_result.document_count)
print("Chunk count:", verification_result.chunk_count)
print("Embedding length:", verification_result.embedding_length)

Document count: 19
Chunk count: 19
Embedding length: 768


In [28]:
from google.api_core.exceptions import ResourceExhausted

QUERY_EMBEDDING_CACHE = {}


def sleep_with_backoff(
    attempt: int,
    *,
    base_seconds: int = 8,
    max_seconds: int = 90,
) -> None:
    sleep_seconds = min(
        max_seconds,
        base_seconds * (2 ** (attempt - 1)),
    )

    print(
        f"Resource exhausted / rate limit. Waiting {sleep_seconds} seconds before retry..."
    )

    time.sleep(sleep_seconds)

In [29]:
def get_query_embedding(
    query: str,
    *,
    max_retries: int = 5,
) -> list[float]:
    clean_query = shorten_text_for_embedding(query)

    if clean_query in QUERY_EMBEDDING_CACHE:
        print("Using cached query embedding.")
        return QUERY_EMBEDDING_CACHE[clean_query]

    for attempt in range(1, max_retries + 1):
        try:
            embeddings = text_embedding_model.get_embeddings(
                [clean_query]
            )

            embedding = list(embeddings[0].values)

            QUERY_EMBEDDING_CACHE[clean_query] = embedding

            return embedding

        except ResourceExhausted:
            if attempt >= max_retries:
                raise

            sleep_with_backoff(attempt)

        except Exception as exc:
            message = str(exc).lower()

            if "resource exhausted" in message or "429" in message:
                if attempt >= max_retries:
                    raise

                sleep_with_backoff(attempt)

            else:
                raise

    raise RuntimeError("Could not create query embedding.")


def search_weaviate_chunks(
    query: str,
    *,
    top_k: int = TOP_K_DEFAULT,
) -> list[dict]:
    query_embedding = get_query_embedding(query)

    sql = f"""
    SELECT
      base.chunk_id AS chunk_id,
      base.document_id AS document_id,
      base.document_number AS document_number,
      base.chunk_number AS chunk_number,
      base.global_chunk_number AS global_chunk_number,
      base.source_type AS source_type,
      base.source_title AS source_title,
      base.section_title AS section_title,
      base.chunk_text AS chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{chunk_table_ref}`
      ),
      'embedding',
      (
        SELECT
          @query_embedding AS embedding
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "query_embedding",
                "FLOAT64",
                query_embedding,
            ),
            bigquery.ScalarQueryParameter(
                "top_k",
                "INT64",
                top_k,
            ),
        ]
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "document_id": row.document_id,
            "document_number": row.document_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "source_type": row.source_type,
            "source_title": row.source_title,
            "section_title": row.section_title,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [24]:
test_query = """
Jaka jest różnica między BM25, vector search i hybrid search w Weaviate?
Wyjaśnij to prosto, najlepiej przez analogię do SQL.
"""

search_results = search_weaviate_chunks(
    test_query,
    top_k=8,
)

print("Query:")
print(test_query)

print("\nResults:", len(search_results))

for result in search_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Section:", result["source_title"])
    print("Chunk:", result["global_chunk_number"])
    print(result["chunk_text"][:900])

Query:

Jaka jest różnica między BM25, vector search i hybrid search w Weaviate?
Wyjaśnij to prosto, najlepiej przez analogię do SQL.


Results: 8
Distance: 0.1915
Section: 16. Weaviate and SQL mental model
Chunk: 17
## 16. Weaviate and SQL mental model SQL database: - table - row - column - foreign key - SQL query Weaviate: - collection - object - property - reference - vector, BM25, hybrid query The analogy is useful but not perfect. Weaviate is optimized for search and AI retrieval, not traditional transactional SQL workloads.
Distance: 0.2128
Section: 7. Hybrid search
Chunk: 8
## 7. Hybrid search Hybrid search combines BM25 keyword search and vector search. This is often better than using only one of them because: - BM25 catches exact terms - vector search catches semantic similarity - hybrid search balances both signals In Weaviate, hybrid search is often useful for RAG, product search, documentation search, and support assistants. The alpha parameter controls the balance: - alpha

In [31]:
def build_rag_context(
    results: list[dict],
) -> str:
    parts = []

    for index, result in enumerate(
        results,
        start=1,
    ):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Chunk ID: {result['chunk_id']}",
                    f"Section: {result['source_title']}",
                    f"Chunk number: {result['global_chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [25]:
def answer_weaviate_question(
    question: str,
    *,
    top_k: int = TOP_K_DEFAULT,
) -> dict:
    results = search_weaviate_chunks(
        question,
        top_k=top_k,
    )

    context = build_rag_context(results)

    prompt = f"""
You are Weaviate Voice Mentor.

Answer in Polish by default.

You help a Python backend learner understand Weaviate, vector databases,
embeddings, semantic search, BM25, hybrid search, RAG, collections,
objects, references, chunks, and Weaviate Cloud.

Use only the retrieved context.

User question:
{question}

Retrieved context:
{context}

Answer style:
- explain simply
- use practical examples
- use SQL table/row analogies when helpful
- mention SOURCE numbers when using evidence
- do not invent information outside the context
- if the context is insufficient, say what is missing
- avoid long theory unless the user asks for it
"""

    response = google_vertex_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return {
        "question": question,
        "answer": response.text,
        "search_results": results,
        "rag_context": context,
    }

In [32]:
single_test_result = answer_weaviate_question(
    "Czym jest kolekcja w Weaviate?",
    top_k=5,
)

print(single_test_result["answer"])

Using cached query embedding.
W Weaviate, **kolekcja** to główny pojemnik na obiekty (dane) (SOURCE 1).

Możesz sobie wyobrazić kolekcję jako odpowiednik **tabeli w tradycyjnej bazie danych SQL** (SOURCE 2, SOURCE 4).

Kolekcja definiuje m.in. (SOURCE 1):
*   jej nazwę,
*   właściwości (pomyśl o nich jak o kolumnach),
*   typy danych dla tych właściwości,
*   konfigurację wektoryzatora (czyli jak Weaviate będzie tworzył wektory dla danych w tej kolekcji),
*   konfigurację modułu generatywnego,
*   właściwości referencyjne (czyli odnośniki do innych kolekcji, jak klucze obce w SQL).

**Przykład praktyczny:**
Możesz stworzyć kolekcję o nazwie `Product` (Produkt) z właściwościami takimi jak `title` (tytuł), `content` (treść), `category` (kategoria), `price` (cena) czy `rating` (ocena) (SOURCE 1). Inne przykładowe kolekcje to `Article` (Artykuł) czy `Question` (Pytanie) (SOURCE 1).

W każdej kolekcji znajdują się poszczególne **obiekty**, które są pojedynczymi rekordami (jak wiersze w tabe

In [33]:
test_questions = [
    "Co to jest kolekcja w Weaviate i jak to porównać do tabeli SQL?",
    "Czym się różni BM25 od vector search?",
    "Kiedy używać hybrid search w Weaviate?",
    "Czy zamknięcie klienta usuwa dane z Weaviate?",
    "Jak działa RAG z Weaviate?",
]

test_answer_rows = []

for question in test_questions:
    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    result = answer_weaviate_question(
        question,
        top_k=8,
    )

    print("\nANSWER:")
    print(result["answer"])

    used_chunk_ids = [
        item["chunk_id"]
        for item in result["search_results"]
    ]

    used_source_titles = sorted(
        {
            item["source_title"]
            for item in result["search_results"]
        }
    )

    test_answer_rows.append(
        {
            "answer_id": str(uuid4()),
            "question": question,
            "answer": result["answer"],
            "used_chunk_ids": used_chunk_ids,
            "used_source_titles": used_source_titles,
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
    )


QUESTION:
Co to jest kolekcja w Weaviate i jak to porównać do tabeli SQL?

ANSWER:
W Weaviate, **kolekcja** to główny kontener na obiekty (dane), które przechowujesz [SOURCE 1]. Można to sobie wyobrazić jako schemat dla Twoich danych, ponieważ definiuje on [SOURCE 1]:
*   nazwę kolekcji
*   właściwości (pola)
*   typy danych dla tych właściwości
*   konfigurację wektoryzatora (jak dane są zamieniane na wektory)
*   konfigurację modułu generatywnego (jeśli używasz AI do generowania tekstu)
*   właściwości referencyjne (czyli odnośniki do innych obiektów/kolekcji)
*   opcjonalne zachowanie indeksowania

Przykładowe kolekcje to `Artykuł`, `Produkt` czy `RecenzjaSprzętu` [SOURCE 1].

### Jak porównać kolekcję do tabeli SQL?

Analogia jest bardzo przydatna [SOURCE 3, SOURCE 7]:
*   **Kolekcja w Weaviate jest podobna do tabeli w bazie danych SQL.** [SOURCE 3]
*   **Obiekt w Weaviate jest podobny do wiersza (rekordu) w tabeli SQL.** [SOURCE 3]
*   **Właściwość obiektu w Weaviate jest podobna

In [34]:
test_answer_load_result = batch_load_rows_to_bigquery(
    rows=test_answer_rows,
    table_ref=test_answer_table_ref,
    schema=test_answer_schema,
    table_name=TEST_ANSWER_TABLE_ID,
    run_id=run_id,
)

print(test_answer_load_result)

Loaded table: weaviate_mentor_test_answers
Rows: 5
GCS URI: gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_test_answers.ndjson
Bytes: 12153
Seconds: 3.1868
{'table_name': 'weaviate_mentor_test_answers', 'gcs_uri': 'gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_test_answers.ndjson', 'row_count': 5, 'source_bytes': 12153, 'load_seconds': 3.1868, 'job_id': '12ff8e8b-044d-4b5f-8ce5-8efde9f69993'}


In [36]:
backend_main_py = f'''
from __future__ import annotations

import os
from typing import Any

import vertexai
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from google import genai
from google.cloud import bigquery
from pydantic import BaseModel, Field
from vertexai.language_models import TextEmbeddingModel


PROJECT_ID = os.environ.get("PROJECT_ID", "{PROJECT_ID}")
LOCATION = os.environ.get("LOCATION", "{LOCATION}")
DATASET_ID = os.environ.get("DATASET_ID", "{DATASET_ID}")
CHUNK_TABLE_ID = os.environ.get("CHUNK_TABLE_ID", "{CHUNK_TABLE_ID}")
TEXT_MODEL = os.environ.get("TEXT_MODEL", "{TEXT_MODEL}")
TEXT_EMBEDDING_MODEL = os.environ.get("TEXT_EMBEDDING_MODEL", "{TEXT_EMBEDDING_MODEL}")
ALLOWED_ORIGIN = os.environ.get("ALLOWED_ORIGIN", "{ALLOWED_ORIGIN}")
TOP_K_DEFAULT = int(os.environ.get("TOP_K_DEFAULT", "{TOP_K_DEFAULT}"))

CHUNK_TABLE_REF = f"{{PROJECT_ID}}.{{DATASET_ID}}.{{CHUNK_TABLE_ID}}"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
)

embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL,
)


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    top_k: int = Field(default=TOP_K_DEFAULT, ge=1, le=20)


class SourceChunk(BaseModel):
    chunk_id: str
    source_title: str
    chunk_number: int
    distance: float
    preview: str


class ChatResponse(BaseModel):
    answer: str
    sources: list[SourceChunk]


app = FastAPI(
    title="Weaviate Voice Mentor RAG API",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        ALLOWED_ORIGIN,
        "http://localhost:3000",
        "http://localhost:5173",
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)


def shorten_text_for_embedding(
    text: str,
    max_chars: int = 3000,
) -> str:
    clean_text = " ".join(text.split())

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_query_embedding(
    query: str,
) -> list[float]:
    embeddings = embedding_model.get_embeddings(
        [
            shorten_text_for_embedding(query)
        ]
    )

    return list(embeddings[0].values)


def search_chunks(
    query: str,
    top_k: int,
) -> list[dict[str, Any]]:
    query_embedding = get_query_embedding(query)

    sql = f"""
    SELECT
      chunk_id,
      document_id,
      document_number,
      chunk_number,
      global_chunk_number,
      source_type,
      source_title,
      section_title,
      chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{{CHUNK_TABLE_REF}}`
      ),
      'embedding',
      (
        SELECT
          @query_embedding AS embedding
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "query_embedding",
                "FLOAT64",
                query_embedding,
            ),
            bigquery.ScalarQueryParameter(
                "top_k",
                "INT64",
                top_k,
            ),
        ]
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
    )

    return [
        {{
            "chunk_id": row.chunk_id,
            "document_id": row.document_id,
            "document_number": row.document_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "source_type": row.source_type,
            "source_title": row.source_title,
            "section_title": row.section_title,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }}
        for row in query_job
    ]


def build_rag_context(
    chunks: list[dict[str, Any]],
) -> str:
    parts = []

    for index, chunk in enumerate(chunks, start=1):
        parts.append(
            "\\n".join(
                [
                    f"[SOURCE {{index}}]",
                    f"Chunk ID: {{chunk['chunk_id']}}",
                    f"Section: {{chunk['source_title']}}",
                    f"Distance: {{chunk['distance']:.4f}}",
                    f"Text: {{chunk['chunk_text']}}",
                ]
            )
        )

    return "\\n\\n---\\n\\n".join(parts)


def generate_answer(
    question: str,
    chunks: list[dict[str, Any]],
) -> str:
    context = build_rag_context(chunks)

    prompt = f"""
You are Weaviate Voice Mentor.

Answer in Polish by default.

You help a Python backend learner understand Weaviate, vector databases,
embeddings, semantic search, BM25, hybrid search, RAG, collections,
objects, references, chunks, and Weaviate Cloud.

Use only the retrieved context.

User question:
{{question}}

Retrieved context:
{{context}}

Answer style:
- explain simply
- use practical examples
- use SQL table/row analogies when helpful
- mention SOURCE numbers when using evidence
- do not invent information outside the context
- if the context is insufficient, say what is missing
- avoid long theory unless the user asks for it
"""

    response = genai_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return response.text


@app.get("/")
def root() -> dict[str, str]:
    return {{
        "status": "ok",
        "service": "Weaviate Voice Mentor RAG API",
    }}


@app.get("/health")
def health() -> dict[str, str]:
    return {{
        "status": "healthy",
        "project_id": PROJECT_ID,
        "dataset_id": DATASET_ID,
        "chunk_table_id": CHUNK_TABLE_ID,
    }}


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest) -> ChatResponse:
    try:
        chunks = search_chunks(
            request.message,
            top_k=request.top_k,
        )

        answer = generate_answer(
            request.message,
            chunks,
        )

        sources = [
            SourceChunk(
                chunk_id=chunk["chunk_id"],
                source_title=chunk["source_title"],
                chunk_number=int(chunk["global_chunk_number"]),
                distance=float(chunk["distance"]),
                preview=chunk["chunk_text"][:260],
            )
            for chunk in chunks
        ]

        return ChatResponse(
            answer=answer,
            sources=sources,
        )

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Chat request failed: {{exc}}",
        ) from exc
'''

In [37]:
backend_requirements_txt = """
fastapi==0.115.6
uvicorn[standard]==0.34.0
google-cloud-bigquery==3.27.0
google-cloud-aiplatform==1.75.0
google-genai==0.5.0
pydantic==2.10.4
""".strip()

In [38]:
backend_dockerfile = """
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8080

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
""".strip()

In [39]:
backend_cloudbuild_yaml = f"""
steps:
  - name: gcr.io/cloud-builders/docker
    args:
      - build
      - -t
      - gcr.io/{PROJECT_ID}/weaviate-voice-mentor-rag
      - .

  - name: gcr.io/cloud-builders/docker
    args:
      - push
      - gcr.io/{PROJECT_ID}/weaviate-voice-mentor-rag

  - name: gcr.io/google.com/cloudsdktool/cloud-sdk
    entrypoint: gcloud
    args:
      - run
      - deploy
      - weaviate-voice-mentor-rag
      - --image
      - gcr.io/{PROJECT_ID}/weaviate-voice-mentor-rag
      - --region
      - {LOCATION}
      - --platform
      - managed
      - --allow-unauthenticated
      - --set-env-vars
      - PROJECT_ID={PROJECT_ID},LOCATION={LOCATION},DATASET_ID={DATASET_ID},CHUNK_TABLE_ID={CHUNK_TABLE_ID},TEXT_MODEL={TEXT_MODEL},TEXT_EMBEDDING_MODEL={TEXT_EMBEDDING_MODEL},ALLOWED_ORIGIN={ALLOWED_ORIGIN},TOP_K_DEFAULT={TOP_K_DEFAULT}

images:
  - gcr.io/{PROJECT_ID}/weaviate-voice-mentor-rag
""".strip()

In [40]:
deploy_commands_sh = f"""
#!/usr/bin/env bash
set -euo pipefail

gcloud config set project {PROJECT_ID}

gcloud services enable \\
  run.googleapis.com \\
  cloudbuild.googleapis.com \\
  artifactregistry.googleapis.com \\
  aiplatform.googleapis.com \\
  bigquery.googleapis.com

gcloud builds submit \\
  --config cloudbuild.yaml \\
  .

gcloud run services describe weaviate-voice-mentor-rag \\
  --region {LOCATION} \\
  --format='value(status.url)'
""".strip()

print(deploy_commands_sh)

#!/usr/bin/env bash
set -euo pipefail

gcloud config set project leafy-guide-497515-m4

gcloud services enable \
  run.googleapis.com \
  cloudbuild.googleapis.com \
  artifactregistry.googleapis.com \
  aiplatform.googleapis.com \
  bigquery.googleapis.com

gcloud builds submit \
  --config cloudbuild.yaml \
  .

gcloud run services describe weaviate-voice-mentor-rag \
  --region us-central1 \
  --format='value(status.url)'


In [41]:
widget_html = """
<div id="weaviate-mentor-widget">
  <button id="wm-open-button">Ask Weaviate Mentor</button>

  <div id="wm-panel" style="display:none;">
    <div id="wm-header">
      <strong>Weaviate Voice Mentor</strong>
      <button id="wm-close-button">×</button>
    </div>

    <div id="wm-messages"></div>

    <div id="wm-controls">
      <textarea id="wm-input" placeholder="Zapytaj o Weaviate..."></textarea>
      <button id="wm-send-button">Send</button>
      <button id="wm-mic-button">🎙️</button>
      <button id="wm-speak-button">🔊</button>
    </div>
  </div>
</div>

<style>
  #weaviate-mentor-widget {
    position: fixed;
    right: 24px;
    bottom: 24px;
    z-index: 9999;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  }

  #wm-open-button {
    border: none;
    border-radius: 999px;
    padding: 14px 18px;
    background: #111827;
    color: white;
    box-shadow: 0 12px 28px rgba(0,0,0,0.24);
    cursor: pointer;
  }

  #wm-panel {
    width: 360px;
    height: 520px;
    background: #0f172a;
    color: white;
    border-radius: 18px;
    overflow: hidden;
    box-shadow: 0 22px 60px rgba(0,0,0,0.35);
    border: 1px solid rgba(255,255,255,0.12);
  }

  #wm-header {
    height: 52px;
    padding: 0 14px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    background: #111827;
    border-bottom: 1px solid rgba(255,255,255,0.12);
  }

  #wm-close-button {
    background: transparent;
    color: white;
    border: none;
    font-size: 24px;
    cursor: pointer;
  }

  #wm-messages {
    height: 350px;
    overflow-y: auto;
    padding: 14px;
    display: flex;
    flex-direction: column;
    gap: 10px;
  }

  .wm-message {
    padding: 10px 12px;
    border-radius: 12px;
    line-height: 1.45;
    font-size: 14px;
    white-space: pre-wrap;
  }

  .wm-user {
    align-self: flex-end;
    background: #2563eb;
  }

  .wm-bot {
    align-self: flex-start;
    background: #1f2937;
  }

  #wm-controls {
    padding: 12px;
    display: grid;
    grid-template-columns: 1fr auto auto auto;
    gap: 8px;
    background: #111827;
  }

  #wm-input {
    resize: none;
    height: 54px;
    border-radius: 10px;
    border: 1px solid rgba(255,255,255,0.16);
    background: #020617;
    color: white;
    padding: 8px;
  }

  #wm-send-button,
  #wm-mic-button,
  #wm-speak-button {
    border: none;
    border-radius: 10px;
    padding: 0 10px;
    background: #334155;
    color: white;
    cursor: pointer;
  }
</style>

<script>
  window.WEAVIATE_MENTOR_API_URL = "PASTE_CLOUD_RUN_URL_HERE";
</script>
<script src="./weaviate-mentor-widget.js"></script>
""".strip()

In [42]:
widget_js = """
(function () {
  const openButton = document.getElementById("wm-open-button");
  const closeButton = document.getElementById("wm-close-button");
  const panel = document.getElementById("wm-panel");
  const messages = document.getElementById("wm-messages");
  const input = document.getElementById("wm-input");
  const sendButton = document.getElementById("wm-send-button");
  const micButton = document.getElementById("wm-mic-button");
  const speakButton = document.getElementById("wm-speak-button");

  let lastAnswer = "";

  function addMessage(text, role) {
    const div = document.createElement("div");
    div.className = "wm-message " + (role === "user" ? "wm-user" : "wm-bot");
    div.textContent = text;
    messages.appendChild(div);
    messages.scrollTop = messages.scrollHeight;
  }

  async function sendMessage() {
    const message = input.value.trim();

    if (!message) {
      return;
    }

    input.value = "";
    addMessage(message, "user");
    addMessage("Myślę...", "bot");

    try {
      const response = await fetch(window.WEAVIATE_MENTOR_API_URL + "/chat", {
        method: "POST",
        headers: {
          "Content-Type": "application/json"
        },
        body: JSON.stringify({
          message: message,
          top_k: 8
        })
      });

      if (!response.ok) {
        throw new Error("HTTP " + response.status);
      }

      const data = await response.json();

      const thinkingNode = messages.lastChild;
      thinkingNode.textContent = data.answer;

      lastAnswer = data.answer;
    } catch (error) {
      const thinkingNode = messages.lastChild;
      thinkingNode.textContent = "Błąd połączenia z Weaviate Mentor API: " + error.message;
    }
  }

  function speakLastAnswer() {
    if (!lastAnswer) {
      return;
    }

    if (!window.speechSynthesis) {
      alert("Speech synthesis is not supported in this browser.");
      return;
    }

    const utterance = new SpeechSynthesisUtterance(lastAnswer);
    utterance.lang = "pl-PL";
    window.speechSynthesis.cancel();
    window.speechSynthesis.speak(utterance);
  }

  function startSpeechRecognition() {
    const SpeechRecognition =
      window.SpeechRecognition || window.webkitSpeechRecognition;

    if (!SpeechRecognition) {
      alert("Speech recognition is not supported in this browser.");
      return;
    }

    const recognition = new SpeechRecognition();
    recognition.lang = "pl-PL";
    recognition.interimResults = false;
    recognition.maxAlternatives = 1;

    recognition.onresult = function (event) {
      input.value = event.results[0][0].transcript;
    };

    recognition.start();
  }

  openButton.addEventListener("click", function () {
    panel.style.display = "block";
    openButton.style.display = "none";
  });

  closeButton.addEventListener("click", function () {
    panel.style.display = "none";
    openButton.style.display = "block";
  });

  sendButton.addEventListener("click", sendMessage);
  speakButton.addEventListener("click", speakLastAnswer);
  micButton.addEventListener("click", startSpeechRecognition);

  input.addEventListener("keydown", function (event) {
    if (event.key === "Enter" && !event.shiftKey) {
      event.preventDefault();
      sendMessage();
    }
  });
})();
""".strip()

In [46]:
backend_files = {
    "main.py": backend_main_py.strip(),
    "requirements.txt": backend_requirements_txt.strip(),
    "Dockerfile": backend_dockerfile.strip(),
    "cloudbuild.yaml": backend_cloudbuild_yaml.strip(),
    "deploy.sh": deploy_commands_sh.strip(),
    "widget.html": widget_html.strip(),
    "weaviate-mentor-widget.js": widget_js.strip(),
}

zip_buffer = BytesIO()

with zipfile.ZipFile(
    zip_buffer,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:
    for filename, content in backend_files.items():
        zip_file.writestr(
            filename,
            content,
        )

zip_buffer.seek(0)

backend_package_blob_name = (
    f"{GCS_BACKEND_PREFIX}/"
    f"{run_id}/"
    "weaviate_mentor_backend_package.zip"
)

backend_package_gcs_uri = upload_bytes_to_gcs(
    zip_buffer.getvalue(),
    blob_name=backend_package_blob_name,
    content_type="application/zip",
)

print("Backend package saved to:")
print(backend_package_gcs_uri)
print("Package bytes:", len(zip_buffer.getvalue()))

Backend package saved to:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/backend/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_backend_package.zip
Package bytes: 6194


In [47]:
backend_asset_rows = []

for filename, content in backend_files.items():
    if filename.endswith(".py"):
        content_type = "text/x-python"
        asset_type = "backend_code"

    elif filename.endswith(".js"):
        content_type = "application/javascript"
        asset_type = "widget_code"

    elif filename.endswith(".html"):
        content_type = "text/html"
        asset_type = "widget_code"

    elif filename.endswith(".yaml") or filename.endswith(".yml"):
        content_type = "application/x-yaml"
        asset_type = "deployment_code"

    elif filename.endswith(".sh"):
        content_type = "text/x-shellscript"
        asset_type = "deployment_code"

    elif filename == "Dockerfile":
        content_type = "text/plain"
        asset_type = "deployment_code"

    else:
        content_type = "text/plain"
        asset_type = "backend_code"

    blob_name = (
        f"{GCS_BACKEND_PREFIX}/"
        f"{run_id}/"
        f"{filename}"
    )

    gcs_uri = upload_text_to_gcs(
        content,
        blob_name=blob_name,
        content_type=content_type,
    )

    backend_asset_rows.append(
        {
            "asset_id": str(uuid4()),
            "asset_type": asset_type,
            "filename": filename,
            "gcs_uri": gcs_uri,
            "content_type": content_type,
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
    )

backend_asset_rows.append(
    {
        "asset_id": str(uuid4()),
        "asset_type": "backend_package",
        "filename": "weaviate_mentor_backend_package.zip",
        "gcs_uri": backend_package_gcs_uri,
        "content_type": "application/zip",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
)

backend_asset_load_result = batch_load_rows_to_bigquery(
    rows=backend_asset_rows,
    table_ref=backend_asset_table_ref,
    schema=backend_asset_schema,
    table_name=BACKEND_ASSET_TABLE_ID,
    run_id=run_id,
)

print("Backend assets uploaded and loaded to BigQuery.")
print("Backend asset rows:", len(backend_asset_rows))
print(backend_asset_load_result)

Loaded table: weaviate_mentor_backend_assets
Rows: 8
GCS URI: gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_backend_assets.ndjson
Bytes: 2700
Seconds: 1.9355
Backend assets uploaded and loaded to BigQuery.
Backend asset rows: 8
{'table_name': 'weaviate_mentor_backend_assets', 'gcs_uri': 'gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_backend_assets.ndjson', 'row_count': 8, 'source_bytes': 2700, 'load_seconds': 1.9355, 'job_id': 'a0b74075-5e03-4217-a521-8c940f6742cd'}


In [48]:
print(backend_main_py[:5000])


from __future__ import annotations

import os
from typing import Any

import vertexai
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from google import genai
from google.cloud import bigquery
from pydantic import BaseModel, Field
from vertexai.language_models import TextEmbeddingModel


PROJECT_ID = os.environ.get("PROJECT_ID", "leafy-guide-497515-m4")
LOCATION = os.environ.get("LOCATION", "us-central1")
DATASET_ID = os.environ.get("DATASET_ID", "weaviate_mentor_rag_backend")
CHUNK_TABLE_ID = os.environ.get("CHUNK_TABLE_ID", "weaviate_mentor_chunks")
TEXT_MODEL = os.environ.get("TEXT_MODEL", "gemini-2.5-flash")
TEXT_EMBEDDING_MODEL = os.environ.get("TEXT_EMBEDDING_MODEL", "text-embedding-005")
ALLOWED_ORIGIN = os.environ.get("ALLOWED_ORIGIN", "https://weaviate-voice-mentor.lovable.app")
TOP_K_DEFAULT = int(os.environ.get("TOP_K_DEFAULT", "8"))

CHUNK_TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"

vertexai.init(
    project=PRO

In [49]:
print(widget_html[:3000])
print("\n" + "=" * 100 + "\n")
print(widget_js[:4000])

<div id="weaviate-mentor-widget">
  <button id="wm-open-button">Ask Weaviate Mentor</button>

  <div id="wm-panel" style="display:none;">
    <div id="wm-header">
      <strong>Weaviate Voice Mentor</strong>
      <button id="wm-close-button">×</button>
    </div>

    <div id="wm-messages"></div>

    <div id="wm-controls">
      <textarea id="wm-input" placeholder="Zapytaj o Weaviate..."></textarea>
      <button id="wm-send-button">Send</button>
      <button id="wm-mic-button">🎙️</button>
      <button id="wm-speak-button">🔊</button>
    </div>
  </div>
</div>

<style>
  #weaviate-mentor-widget {
    position: fixed;
    right: 24px;
    bottom: 24px;
    z-index: 9999;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  }

  #wm-open-button {
    border: none;
    border-radius: 999px;
    padding: 14px 18px;
    background: #111827;
    color: white;
    box-shadow: 0 12px 28px rgba(0,0,0,0.24);
    cursor: pointer;
  }

  #wm-panel {
    widt

In [50]:
deployment_notes = f"""
Deployment plan:

1. Download or copy these files from GCS:
   - main.py
   - requirements.txt
   - Dockerfile
   - cloudbuild.yaml
   - deploy.sh

2. Put them into one local folder, for example:
   cloud_run_weaviate_mentor/

3. Run:
   bash deploy.sh

4. After deployment, Cloud Run prints a public HTTPS URL.

5. In widget.html replace:
   PASTE_CLOUD_RUN_URL_HERE

   with your Cloud Run service URL, for example:
   https://weaviate-voice-mentor-rag-xxxxx.a.run.app

6. Paste widget.html and weaviate-mentor-widget.js into Lovable or host the JS file separately.

Backend package:
{backend_package_gcs_uri}

Knowledge base:
{knowledge_gcs_uri}
"""

print(deployment_notes)


Deployment plan:

1. Download or copy these files from GCS:
   - main.py
   - requirements.txt
   - Dockerfile
   - cloudbuild.yaml
   - deploy.sh

2. Put them into one local folder, for example:
   cloud_run_weaviate_mentor/

3. Run:
   bash deploy.sh

4. After deployment, Cloud Run prints a public HTTPS URL.

5. In widget.html replace:
   PASTE_CLOUD_RUN_URL_HERE

   with your Cloud Run service URL, for example:
   https://weaviate-voice-mentor-rag-xxxxx.a.run.app

6. Paste widget.html and weaviate-mentor-widget.js into Lovable or host the JS file separately.

Backend package:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/backend/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_backend_package.zip

Knowledge base:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_knowledge_base.md



In [51]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "vertex_ai_location": LOCATION,
    "bigquery_location": dataset.location,
    "bucket_location": BUCKET_LOCATION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": "68_w_google_cloud_weaviate_mentor_rag_backend.ipynb",
    "run_id": run_id,
    "models": {
        "text_model": TEXT_MODEL,
        "planning_model": PLANNING_MODEL,
        "text_embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "bigquery": {
        "dataset": DATASET_ID,
        "document_table": document_table_ref,
        "chunk_table": chunk_table_ref,
        "test_answer_table": test_answer_table_ref,
        "backend_asset_table": backend_asset_table_ref,
    },
    "cloud_storage": {
        "knowledge_gcs_uri": knowledge_gcs_uri,
        "backend_package_gcs_uri": backend_package_gcs_uri,
        "backend_prefix": f"gs://{BUCKET_NAME}/{GCS_BACKEND_PREFIX}/{run_id}",
        "local_files_saved": False,
    },
    "counts": {
        "documents": len(document_rows),
        "chunks": len(chunk_rows),
        "test_answers": len(test_answer_rows),
        "backend_assets": len(backend_asset_rows),
    },
    "performance": {
        "embedding_seconds": round(embedding_seconds, 2),
    },
    "deployment_notes": deployment_notes,
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{run_id}/"
    "summary.json"
)

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=summary_blob_name,
    content_type="application/json",
)

print("Summary saved to:")
print(summary_gcs_uri)

Summary saved to:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/summaries/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/summary.json


In [52]:
sql = f"""
SELECT
  asset_type,
  filename,
  gcs_uri,
  content_type,
  created_at
FROM `{backend_asset_table_ref}`
ORDER BY
  asset_type,
  filename
"""

backend_assets_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

backend_assets_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,asset_type,filename,gcs_uri,content_type,created_at
0,backend_code,main.py,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/x-python,2026-07-15 15:50:10.271229+00:00
1,backend_code,requirements.txt,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/plain,2026-07-15 15:50:10.548932+00:00
2,backend_package,weaviate_mentor_backend_package.zip,gs://leafy-guide-497515-m4-vector-assets/weavi...,application/zip,2026-07-15 15:50:11.845259+00:00
3,deployment_code,Dockerfile,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/plain,2026-07-15 15:50:10.797983+00:00
4,deployment_code,cloudbuild.yaml,gs://leafy-guide-497515-m4-vector-assets/weavi...,application/x-yaml,2026-07-15 15:50:11.065531+00:00
5,deployment_code,deploy.sh,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/x-shellscript,2026-07-15 15:50:11.323647+00:00
6,widget_code,weaviate-mentor-widget.js,gs://leafy-guide-497515-m4-vector-assets/weavi...,application/javascript,2026-07-15 15:50:11.844948+00:00
7,widget_code,widget.html,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/html,2026-07-15 15:50:11.590559+00:00


In [53]:
print("Weaviate Mentor RAG Backend notebook completed.")
print("=" * 100)

print("Run ID:", run_id)

print("\nKnowledge base:")
print(knowledge_gcs_uri)

print("\nBigQuery tables:")
print("-", document_table_ref)
print("-", chunk_table_ref)
print("-", test_answer_table_ref)
print("-", backend_asset_table_ref)

print("\nCounts:")
print("Documents:", len(document_rows))
print("Chunks:", len(chunk_rows))
print("Test answers:", len(test_answer_rows))
print("Backend assets:", len(backend_asset_rows))

print("\nCloud Run backend package:")
print(backend_package_gcs_uri)

print("\nSummary:")
print(summary_gcs_uri)

print("\nNext deployment step:")
print("Download/copy the generated backend files, then run:")
print("bash deploy.sh")

print("\nNo local files were saved by this notebook.")

Weaviate Mentor RAG Backend notebook completed.
Run ID: b3c00ed4-5393-4261-bd1f-c1df30fe8cf7

Knowledge base:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/source/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_knowledge_base.md

BigQuery tables:
- leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_documents
- leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_chunks
- leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_test_answers
- leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_backend_assets

Counts:
Documents: 19
Chunks: 19
Test answers: 5
Backend assets: 8

Cloud Run backend package:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/backend/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/weaviate_mentor_backend_package.zip

Summary:
gs://leafy-guide-497515-m4-vector-assets/weaviate-mentor-rag/summaries/b3c00ed4-5393-4261-bd1f-c1df30fe8cf7/summary.json

Next deployment step:
Download/copy the generated bac